# Business Objective
This project simulates a Nigerian digital lending company seeking to build a credit risk model to predict the likelihood of borrower default.

The objective is to develop a machine learning system that can:

- Assess borrower creditworthiness at loan application time

- Predict probability of default (PD)

- Support risk-based pricing decisions

- Reduce non-performing loans

The model will use borrower demographics, loan characteristics, and credit history features to predict whether a loan will be fully repaid or defaulted.

Although the dataset originates from Lending Club (U.S.), it is adapted to simulate a Nigerian fintech lending environment.

## Business Problem Framing

This is a binary classification problem:

- 1 → Default (high-risk borrower)

- 0 → Fully Paid (low-risk borrower)

The output will be a probability score that can be integrated into a loan approval API.

## Dataset Overview

The dataset is derived from Lending Club historical loan data.

Original dataset:

- ~890,000 observations

- ~145 variables

- Multiple loan statuses

For this project:

- A 100,000 observation random sample was created for computational efficiency.

- Only finalized loans will be used for modeling.

The dataset contains:

**Borrower Information**

- Employment length

- Home ownership status

- Annual income

- State

**Loan Characteristics**

- Loan amount

- Interest rate

- Term

- Installment

- Loan purpose

- Credit grade

**Credit History**

- Debt-to-income ratio (DTI)

- Revolving balance

- Delinquencies

- Total credit accounts

**Target Variable**

- `loan_status` **(to be converted into binary target)**

In [1]:
# import the necessary libraries
import numpy as np
import pandas as pd

In [2]:
# load dataset
df_loan_raw = pd.read_csv(r'C:\Users\osaze\Desktop\nigerian_lending\data\raw\loan_sample.csv', 
                          low_memory=False)
df_loan_raw.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,35000,35000,34950.0,36 months,12.12,1164.51,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,15000,15000,15000.0,60 months,13.49,345.08,C,C2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,8.99,158.98,B,B1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,33600,33600,33600.0,36 months,10.91,1098.59,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,23000,23000,23000.0,36 months,7.89,719.57,A,A5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# inspect the dataset using info() method
df_loan_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Columns: 145 entries, id to settlement_term
dtypes: float64(105), int64(4), str(36)
memory usage: 110.6 MB


***The dataset contains many null values, dirty string columns requiring cleaning, and categorical variables needing encoding. Before addressing these, we'll first select relevant features from the 145 columns—some may be post-outcome variables that cause data leakage, creating artificially impressive but unrealistic model performance.***

In [4]:
# create and print a list of all columns
column_name = list(df_loan_raw.columns)
column_name

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 'dti_joint',
 'verification_status_joint',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_

In [5]:
columns_to_keep = [
    'loan_status','loan_amnt','term','int_rate','installment','grade','sub_grade',
'purpose','application_type','disbursement_method','initial_list_status','emp_length','home_ownership',
'annual_inc','verification_status','addr_state', 'dti','delinq_2yrs','inq_last_6mths','open_acc',
'pub_rec','revol_bal','revol_util','total_acc','pub_rec_bankruptcies','tax_liens','collections_12_mths_ex_med',
'acc_now_delinq','chargeoff_within_12_mths','delinq_amnt','total_rev_hi_lim','bc_util','percent_bc_gt_75',
'avg_cur_bal','total_bal_ex_mort','total_bc_limit','tot_cur_bal','tot_hi_cred_lim','earliest_cr_line',
'mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op','mths_since_recent_inq'
]

In [6]:
# create a subset of the original data that contains only the relevant loans
df_loan_model = df_loan_raw[columns_to_keep]
df_loan_model.head()

,loan_status,loan_amnt,term,int_rate,installment,grade,sub_grade,purpose,application_type,disbursement_method,...,percent_bc_gt_75,avg_cur_bal,total_bal_ex_mort,total_bc_limit,tot_cur_bal,tot_hi_cred_lim,earliest_cr_line,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mths_since_recent_inq
0,Fully Paid,35000,36 months,12.12,1164.51,B,B3,credit_card,Individual,Cash,...,0.0,20612.0,41223.0,4500.0,41223.0,49161.0,Jan-1983,238.0,30.0,12.0
1,Current,15000,60 months,13.49,345.08,C,C2,home_improvement,Individual,Cash,...,0.0,849.0,14440.0,13800.0,14440.0,42400.0,Oct-1991,304.0,10.0,4.0
2,Fully Paid,5000,36 months,8.99,158.98,B,B1,debt_consolidation,Individual,Cash,...,0.0,1410.0,7050.0,5300.0,7050.0,14760.0,May-1989,326.0,14.0,0.0
3,Current,33600,36 months,10.91,1098.59,B,B4,credit_card,Individual,Cash,...,33.3,8392.0,92316.0,13500.0,92316.0,122340.0,Feb-2008,58.0,4.0,4.0
4,Fully Paid,23000,36 months,7.89,719.57,A,A5,debt_consolidation,Individual,Cash,...,0.0,15661.0,37604.0,11500.0,125287.0,152147.0,Feb-1992,291.0,26.0,8.0


In [7]:
# perform overview of the dataset again
df_loan_model.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 42 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   loan_status                 100000 non-null  str    
 1   loan_amnt                   100000 non-null  int64  
 2   term                        100000 non-null  str    
 3   int_rate                    100000 non-null  float64
 4   installment                 100000 non-null  float64
 5   grade                       100000 non-null  str    
 6   sub_grade                   100000 non-null  str    
 7   purpose                     100000 non-null  str    
 8   application_type            100000 non-null  str    
 9   disbursement_method         100000 non-null  str    
 10  initial_list_status         100000 non-null  str    
 11  emp_length                  93447 non-null   str    
 12  home_ownership              100000 non-null  str    
 13  annual_inc                

In [8]:
# first thing first, let us clean the target variable
list(df_loan_model.loan_status.unique())

['Fully Paid',
 'Current',
 'Late (31-120 days)',
 'Charged Off',
 'In Grace Period',
 'Late (16-30 days)',
 'Does not meet the credit policy. Status:Fully Paid',
 'Does not meet the credit policy. Status:Charged Off',
 'Default']

## Target Engineering: `loan_status` → `default_flag`

**Why filter statuses?** Only **finalized loans** have known repayment outcomes. Active loans (*Current*, *Late*) create **label noise** since their final status is unknown.

**Keep (Binary Target):**
- **0 (Good)**: `Fully Paid`, `Does not meet credit policy. Status:Fully Paid`
- **1 (Default)**: `Charged Off`, `Default`, `Does not meet credit policy. Status:Charged Off`

**Drop**: `Current`, `Late (31-120 days)`, `Late (16-30 days)`, `In Grace Period`  
*(Future uncertainty → model unreliability)*

**Result**: Clean binary `default_flag` for credit risk classification. Next: check imbalance.

In [9]:
good_loans_status = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
bad_loans_status = ["Charged Off", "Default", "Does not meet the credit policy:Charged Off"]

# filter the loan_status column to include 
# only the needed columns in the good_loans_status and bad_loan_status list
filtered_df = df_loan_model[df_loan_model.loan_status.isin(good_loans_status + bad_loans_status)]

# create a default flag column
filtered_df['default_flag'] = filtered_df['loan_status'].isin(bad_loans_status).astype(int)

In [10]:
# check the distribution of the target variable
filtered_df.default_flag.value_counts(normalize=True)

default_flag
0    0.798084
1    0.201916
Name: proportion, dtype: float64

*Target shows 80/20 class imbalance (default_flag: 79.8% good loans, 20.2% defaults).*

*Imbalance handling will be applied consistently across all models using class weighting techniques to ensure robust default detection performance regardless of algorithm choice.*

In [11]:
# perform an overview of the filtered dataset
filtered_df.info()

<class 'pandas.DataFrame'>
Index: 57722 entries, 0 to 99999
Data columns (total 43 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   loan_status                 57722 non-null  str    
 1   loan_amnt                   57722 non-null  int64  
 2   term                        57722 non-null  str    
 3   int_rate                    57722 non-null  float64
 4   installment                 57722 non-null  float64
 5   grade                       57722 non-null  str    
 6   sub_grade                   57722 non-null  str    
 7   purpose                     57722 non-null  str    
 8   application_type            57722 non-null  str    
 9   disbursement_method         57722 non-null  str    
 10  initial_list_status         57722 non-null  str    
 11  emp_length                  54332 non-null  str    
 12  home_ownership              57722 non-null  str    
 13  annual_inc                  57721 non-null  flo

In [12]:
# check for null values
filtered_df.isnull().sum()

loan_status                      0
loan_amnt                        0
term                             0
int_rate                         0
installment                      0
grade                            0
sub_grade                        0
purpose                          0
application_type                 0
disbursement_method              0
initial_list_status              0
emp_length                    3390
home_ownership                   0
annual_inc                       1
verification_status              0
addr_state                       0
dti                             18
delinq_2yrs                      2
inq_last_6mths                   2
open_acc                         2
pub_rec                          2
revol_bal                        0
revol_util                      35
total_acc                        2
pub_rec_bankruptcies            52
tax_liens                        3
collections_12_mths_ex_med       5
acc_now_delinq                   2
chargeoff_within_12_

*We'll defer null value imputation until after the train-test split and pipeline creation.*

*This prevents data leakage by ensuring test set statistics (medians, modes) don't influence training transformations.*

*Also, we have to drop the loan_status column since we have the default_flag column now.*

In [13]:
# drop the loan_status column

filtered_df = filtered_df.drop('loan_status', axis=1)

#confirm the column has been dropped
filtered_df.shape

(57722, 42)

In [14]:
# remove the months in the term column
filtered_df['term'] = filtered_df['term'].str.replace(' months', '').astype(int)

In [15]:
# let us check the data if there is any cleaning that still needs to be done
filtered_df.head().T

,0,2,4,7,8
loan_amnt,35000,5000,23000,12000,15000
term,36,36,36,36,36
int_rate,12.12,8.99,7.89,6.97,7.69
installment,1164.51,158.98,719.57,370.37,467.91
grade,B,B,A,A,A
sub_grade,B3,B1,A5,A3,A4
purpose,credit_card,debt_consolidation,debt_consolidation,credit_card,debt_consolidation
application_type,Individual,Individual,Individual,Individual,Individual
disbursement_method,Cash,Cash,Cash,Cash,Cash
initial_list_status,f,f,w,w,f


*From the overview, we can see that we still need to clean the emp_length column and a new column would be created from the earliest_cr_line column that shows the years of credit history*

In [16]:
# check the unique values in the emp_length column
filtered_df.emp_length.unique()

<StringArray>
[  '5 years', '10+ years',    '1 year',   '8 years',  '< 1 year',   '3 years',
   '2 years',   '4 years',         nan,   '6 years',   '7 years',   '9 years']
Length: 12, dtype: str

In [17]:
# convert the emp_length to int and remove the years, + and <

filtered_df['emp_length'] = (filtered_df['emp_length']
                           .str.replace(' years', '', regex=False)
                           .str.replace(' year', '', regex=False)
                           .str.replace('< 1', '0', regex=True)
                           .str.replace(r'[\+]', '', regex=True)
                           .astype(float))

# check the column again to confirm if corrections have been made
filtered_df.emp_length.unique()

array([ 5., 10.,  1.,  8.,  0.,  3.,  2.,  4., nan,  6.,  7.,  9.])

In [18]:
# Create a new column called credit_age_years
filtered_df['earliest_cr_line'] = pd.to_datetime(filtered_df['earliest_cr_line'], format='%b-%Y')
filtered_df['credit_age_years'] = (
    (pd.Timestamp('2026-03-05') - filtered_df['earliest_cr_line']).dt.days / 365.25
)

# Drop original date column
filtered_df = filtered_df.drop('earliest_cr_line', axis=1)

In [19]:
# check the dataset again
filtered_df.head().T


,0,2,4,7,8
loan_amnt,35000,5000,23000,12000,15000
term,36,36,36,36,36
int_rate,12.12,8.99,7.89,6.97,7.69
installment,1164.51,158.98,719.57,370.37,467.91
grade,B,B,A,A,A
sub_grade,B3,B1,A5,A3,A4
purpose,credit_card,debt_consolidation,debt_consolidation,credit_card,debt_consolidation
application_type,Individual,Individual,Individual,Individual,Individual
disbursement_method,Cash,Cash,Cash,Cash,Cash
initial_list_status,f,f,w,w,f


In [20]:
# Convert the cleaned dataframe to a csv file
filtered_df.to_csv(r'C:\Users\osaze\Desktop\nigerian_lending\data\processed\loans_cleaned.csv', 
                   index=False)